# YTAgent — Free 3D Video + Kokoro Narration Worker

This notebook runs Wan 2.1 on Kaggle's free GPU. It connects OUTBOUND to YTAgent over HTTPS, so no Cloudflare tunnel or changing public URL is needed. Turn **Internet on** and select a **GPU** accelerator in Notebook settings. Keep the notebook running while a video is being generated.The worker uses Wan 2.1 for real 3D video scenes and Kokoro 82M for local documentary narration. Both run without paid generation APIs. Turn **Internet on** and select a **GPU** accelerator. Keep the notebook running while a film is being generated.

In [ ]:
import os, subprocess
repo_dir = "/kaggle/working/YTAgent"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/qingremzy/YTAgent.git", repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(repo_dir + "/gpu-worker")
print("Installing worker packages (Python 3.13-compatible pins)...")
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)
print("Installing Kokoro + Misaki source without the incompatible curated-transformers extra...")
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "git+https://github.com/hexgrad/misaki.git"], check=True)
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "git+https://github.com/hexgrad/kokoro.git"], check=True)
print("Installing espeak-ng for Kokoro pronunciation...")
subprocess.run(["apt-get", "-qq", "-y", "update"], check=True)
subprocess.run(["apt-get", "-qq", "-y", "install", "espeak-ng"], check=True)
print("Checking TTS imports...")
subprocess.run(["python", "-c", "from kokoro import KPipeline; import spacy; print('Kokoro and spaCy imports OK', spacy.__version__)"], check=True)
print("Setup complete. No tunnel software is needed.")

In [ ]:
import os
worker_token = os.environ.get("YT_AGENT_WORKER_TOKEN")
if not worker_token:
    try:
        from kaggle_secrets import UserSecretsClient
        worker_token = UserSecretsClient().get_secret("YT_AGENT_WORKER_TOKEN")
        os.environ["YT_AGENT_WORKER_TOKEN"] = worker_token
    except Exception:
        raise RuntimeError("The worker token is not available. Preserve the existing live notebook session, or add YT_AGENT_WORKER_TOKEN under Kaggle Notebook Add-ons → Secrets.")
print("Private worker token loaded. It will not be displayed.")

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Kaggle Notebook settings, select a GPU accelerator.")

In [ ]:
import subprocess, time, os
os.environ["YT_AGENT_SERVER_URL"] = "https://ytagent-production-1038.up.railway.app"
if "worker_process" in globals() and worker_process.poll() is None:
    worker_process.terminate()
    try:
        worker_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        worker_process.kill()
worker_log = open("/tmp/ytagent-worker.log", "w")
worker_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd=os.getcwd(), stdout=worker_log, stderr=subprocess.STDOUT
)
time.sleep(8)
if worker_process.poll() is not None:
    print(open("/tmp/ytagent-worker.log").read()[-4000:])
    raise RuntimeError("GPU worker failed to start. Check the error above.")
print("GPU worker started. It is polling YTAgent securely over HTTPS. No tunnel is required.")

In [ ]:
## Connection status

The worker polls YTAgent over outbound HTTPS, generates ten Wan video scenes, then creates ten Kokoro narration tracks and uploads both. There is no Cloudflare URL to copy and no tunnel to maintain.

## Refresh after a YTAgent code update

Run Cell 1 to pull the latest worker code, Cell 2 to load the private token and check the GPU, then Cell 3 to restart the worker. Keep the same notebook session when possible so the private token remains available. If you need a fresh session, save `YT_AGENT_WORKER_TOKEN` as a Kaggle Notebook Secret first.

## Refresh after a YTAgent code update

If the Kaggle notebook was already running before a YTAgent update, run Cell 1 to pull the latest worker code, then run Cell 3 to restart only the worker. The token cell reuses the existing token in the same live notebook session. Leave the Cloudflare tunnel cell running. Do not restart the Kaggle session unless necessary, because a fresh session creates a new token and requires updating the matching Railway variable.